In [ ]:
import duckdb
import polars as pl
import pandas as pd
import json
import os

In [ ]:
def add_hashed_payload_to_bronze(
    bronze_db_path: str,
    table_name: str,
) -> None:
    with duckdb.connect(bronze_db_path) as bronze_con:
        columns = {
            row[1]
            for row in bronze_con.execute(
                f"PRAGMA table_info('{table_name}')"
            ).fetchall()
        }

        if "hashed_payload" not in columns:
            bronze_con.execute(
                f"""
                ALTER TABLE {table_name}
                ADD COLUMN hashed_payload VARCHAR;
                """
            )

        bronze_con.execute(
            f"""
            UPDATE {table_name}
            SET hashed_payload = md5(
                CAST(payload AS VARCHAR)
            )
            WHERE hashed_payload IS NULL;
            """
        )

        bronze_con.execute(
            f"""
            ALTER TABLE {table_name}
            ALTER COLUMN hashed_payload SET NOT NULL;
            """
        )

In [ ]:
def reorder_hashed_payload_column(
    bronze_db_path: str,
    table_name: str,
) -> None:
    temp_table_name = f"{table_name}_reordered"

    with duckdb.connect(bronze_db_path) as bronze_con:
        bronze_con.execute("BEGIN TRANSACTION")

        try:
            bronze_con.execute(
                f"""
                CREATE TABLE {temp_table_name} (
                    run_id VARCHAR NOT NULL,
                    project_type VARCHAR NOT NULL,
                    project_id VARCHAR NOT NULL,
                    payload JSON NOT NULL,
                    hashed_payload VARCHAR NOT NULL,
                    c_pull_timestamp_utc TIMESTAMPTZ NOT NULL,
                    PRIMARY KEY (
                        run_id,
                        project_type,
                        project_id
                    )
                );
                """
            )

            bronze_con.execute(
                f"""
                INSERT INTO {temp_table_name}
                (
                    run_id,
                    project_type,
                    project_id,
                    payload,
                    hashed_payload,
                    c_pull_timestamp_utc
                )
                SELECT
                    run_id,
                    project_type,
                    project_id,
                    payload,
                    hashed_payload,
                    c_pull_timestamp_utc
                FROM {table_name};
                """
            )

            bronze_con.execute(
                f"""
                DROP TABLE {table_name};
                """
            )

            bronze_con.execute(
                f"""
                ALTER TABLE {temp_table_name}
                RENAME TO {table_name};
                """
            )

            bronze_con.execute("COMMIT")

        except Exception:
            bronze_con.execute("ROLLBACK")
            raise

In [ ]:
reorder_hashed_payload_column(bronze_db_path="/Users/admin/AroTekCodingSpace/Python-Workspace/Minecraft-Data-Platform/data/bronze/dev/https:||api.modrinth.com.duckdb",
                              table_name='modrinth_project_listings')

In [ ]:
add_hashed_payload_to_bronze(bronze_db_path="/Users/admin/AroTekCodingSpace/Python-Workspace/Minecraft-Data-Platform/data/bronze/dev/https:||api.modrinth.com.duckdb",
                             table_name='modrinth_project_listings')

In [ ]:
with duckdb.connect("/Users/admin/AroTekCodingSpace/Python-Workspace/Minecraft-Data-Platform/data/bronze/dev/https:||piston-meta.mojang.com.duckdb") as con:
    df = con.execute("SELECT * FROM ingestion_log").fetch_df()
df

In [ ]:
with duckdb.connect("/Users/admin/AroTekCodingSpace/Python-Workspace/Minecraft-Data-Platform/data/bronze/dev/https:||api.modrinth.com.duckdb") as con:
    df = con.execute("SELECT * FROM ingestion_log").fetch_df()
df

In [ ]:
with duckdb.connect("/Users/admin/AroTekCodingSpace/Python-Workspace/Minecraft-Data-Platform/data/bronze/dev/api.modrinth.com.duckdb") as con:
    df = con.execute("SELECT * FROM modrinth_project_listings LIMIT 10").fetch_df()
df

In [ ]:
duckdb.sql("SELECT version()").show()

In [ ]:
payload = json.loads(df["payload"].iloc[0])

print(
    json.dumps(
        payload,
        indent=4,
    )
)